## Setup
Run notebooks 01–07 first. Install dependencies once from the project root with `python -m pip install -r requirements.txt`.

In [1]:
from pathlib import Path
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate/"data").is_dir() and (candidate/"notebooks").is_dir(): PROJECT_ROOT=candidate; break
else: raise RuntimeError("Start Jupyter inside the project so its root can be found.")
DATA_DIR=PROJECT_ROOT/"data"; INTERMEDIATE_DIR=DATA_DIR/"02_intermediate"; MODEL_DIR=PROJECT_ROOT/"models"; REPORT_DIR=PROJECT_ROOT/"reports"
INTERMEDIATE_DIR.mkdir(parents=True,exist_ok=True); MODEL_DIR.mkdir(parents=True,exist_ok=True); REPORT_DIR.mkdir(parents=True,exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\acer\Desktop\Programs\Projects_for _data_science\Fraud_Detection_Model


# 11 · Monitoring, drift, and feedback

Monitoring checks whether data, score distributions, and outcomes remain useful after launch. This notebook illustrates period comparisons using the historical validation window; it is not a production feed and drift alone does not prove the model is failing.

### Why divide the validation period

The same fixed model scores an earlier and later half of the validation window. A change between them illustrates monitoring calculations without changing the model. It is not a true post-deployment simulation, and both periods come from the data used during model selection.

In [ ]:
train_features = pd.read_pickle(INTERMEDIATE_DIR / "train_features.pkl")
artifact = joblib.load(MODEL_DIR / "best_experiment_model.joblib")
feature_cols = list(artifact["feature_cols"])

ordered = train_features.sort_values("TransactionDT", kind="stable").reset_index(drop=True)
validation = ordered.iloc[int(len(ordered) * 0.8):].copy()
cut_point = len(validation) // 2
periods = {
    "validation_early": validation.iloc[:cut_point].copy(),
    "validation_late": validation.iloc[cut_point:].copy(),
}

for frame in periods.values():
    X_period = frame[feature_cols].replace([np.inf, -np.inf], np.nan)
    if artifact.get("imputer") is not None:
        X_period = artifact["imputer"].transform(X_period)
    if artifact.get("scaler") is not None:
        X_period = artifact["scaler"].transform(X_period)
    frame["score"] = artifact["model"].predict_proba(X_period)[:, 1]

period_bounds = {
    name: (len(frame), frame["TransactionDT"].min(), frame["TransactionDT"].max())
    for name, frame in periods.items()
}
period_bounds

## 1. Data quality and score movement

### Why monitor quality and score summaries together

Missingness can reveal an upstream collection failure; score mean and upper quantiles can reveal changes in what the model is seeing. Fraud rate is included only because labels exist in this historical exercise. In production, label maturity must be checked before comparing outcome rates.

In [ ]:
def data_quality(frame):
    """Summarize completeness and score movement for one period."""
    X_period = frame[feature_cols].replace([np.inf, -np.inf], np.nan)
    return pd.Series({
        "rows": len(frame),
        "missing_feature_cells_pct": X_period.isna().to_numpy().mean(),
        "fraud_rate": frame["isFraud"].mean(),
        "mean_score": frame["score"].mean(),
        "p95_score": frame["score"].quantile(0.95),
    })

quality = pd.DataFrame({
    name: data_quality(frame)
    for name, frame in periods.items()
}).T
quality

## 2. Population Stability Index

PSI summarizes change in a binned distribution. Bins are fixed from the earlier window, with smoothing for empty bins. Treat it as a screening signal, not a significance test or an automatic retraining trigger.

### Why PSI and what its number means

The code bins the later values using quantile boundaries from the earlier period, compares each bin’s share, and sums the divergence. Smoothing prevents a zero-count bin from making the logarithm undefined. PSI is a compact alert signal, not a test of cause or a universal retraining rule; investigate the feature and the pipeline behind a shift.

In [ ]:
def population_stability_index(reference, current, bins=10, epsilon=1e-6):
    """Compare distributions using bins defined by the reference period."""
    reference = pd.Series(reference).dropna().to_numpy()
    current = pd.Series(current).dropna().to_numpy()
    if len(reference) == 0 or len(current) == 0:
        return np.nan

    edges = np.unique(np.quantile(reference, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0  # Too little reference variation to form useful bins.
    edges[0], edges[-1] = -np.inf, np.inf

    reference_share = np.histogram(reference, bins=edges)[0] / len(reference)
    current_share = np.histogram(current, bins=edges)[0] / len(current)
    reference_share = np.clip(reference_share, epsilon, None)
    current_share = np.clip(current_share, epsilon, None)
    return float(np.sum(
        (current_share - reference_share)
        * np.log(current_share / reference_share)
    ))

reference = periods["validation_early"]
current = periods["validation_late"]
feature_candidates = [
    column for column in ["TransactionAmt", "card1_txn_count_1h", "n_missing"]
    if column in reference.columns
]

psi_rows = [{
    "variable": "score",
    "psi": population_stability_index(reference["score"], current["score"]),
}]
for column in feature_candidates:
    psi_rows.append({
        "variable": column,
        "psi": population_stability_index(reference[column], current[column]),
    })

drift_summary = pd.DataFrame(psi_rows)
drift_summary.sort_values("psi", ascending=False)

## 3. Outcome metrics after labels mature

### Why outcome metrics use a threshold

Average precision measures ranking without choosing one cutoff. Precision, recall, and flag rate describe one operating point, so they show the fraud capture and workload trade-off at `0.53`. That threshold is copied as an illustration from the prior cost analysis, not a validated monitoring policy.

In [ ]:
MONITOR_THRESHOLD = 0.53  # Illustrative; validate against the current operating policy.
metric_rows = []

for period_name, frame in periods.items():
    truth = frame["isFraud"].astype(int)
    predicted_flag = (frame["score"] >= MONITOR_THRESHOLD).astype(int)
    metric_rows.append({
        "period": period_name,
        "rows": len(frame),
        "fraud_rate": truth.mean(),
        "average_precision": average_precision_score(truth, frame["score"]),
        "precision_at_threshold": precision_score(
            truth, predicted_flag, zero_division=0
        ),
        "recall_at_threshold": recall_score(
            truth, predicted_flag, zero_division=0
        ),
        "flag_rate": predicted_flag.mean(),
    })

pd.DataFrame(metric_rows).set_index("period")

## 4. Monitoring playbook

Track input schema and missingness, feature freshness, prediction latency/failures, score and action rates, label maturity, fraud capture, false declines, review queue volume, and customer outcomes. Assign owners and alert thresholds based on historical variation. Investigate pipeline and population changes before retraining. Preserve prediction-time features and model versions so delayed labels can be joined without leakage.